# NB4: CD/CI 예측 — 모델 축(arch/scope/level) 입력 제거 ablation

**목적**: 제안한 모델 축을 입력에 넣지 **않고**(또는 일부만 넣고) 데이터셋 메타피처만으로
CD vs CI 학습전략 우열 점수(perf / time / mem)를 예측할 수 있는지, 그리고 성능이 얼마나 하락하는지 확인.

**NB2(`02_ml_modeling.ipynb`) 와의 차이 — 비교 가능하도록 모델·학습 세팅은 전부 동일**
- 입력 X: NB2 = 메타피처 + (arch, scope, level) one-hot.  **NB4 = 메타피처 + `INCLUDE_AXES` 로 고른 축만** (기본 None = 축 전부 제외).
- 데이터 단위: `AGG_FUNC` — `'mean'`/`'median'`=데이터셋당 1행, `None`=집계 없이 combo 단위(48행).
- split: NB2 가 저장한 `best_model.pkl` 의 train/val/test 를 **그대로 로드** → NB2 / NB3 와 동일.
- 평가: TEST 데이터셋 추론 + §6 에서 **`INCLUDE_AXES` 조합([전체축, Arch, Scope, Level, None])을 NB4 내부 재학습**해 perf_score ROC 비교. (전체축 = with-axis = NB2/NB3 동등이라 외부 결과 불필요)

**입력** (NB1 / NB2 산출물 재사용)
- `outputs/nb1/meta_features_full.csv`, `outputs/nb1/feature_groups.csv`
- `outputs/nb2/brickts_wide.csv`, `outputs/nb2/best_model.pkl` (split)

**출력** (`outputs/nb4/`)
- `dataset_level_table.csv` / `inference_test_results.csv` / `inference_test_metrics.csv`
- `axis_ablation_auc.csv` (조합별 perf AUC) / `feature_importance_noaxis.csv` / `model_noaxis.pkl`

## Section 0 — 환경 / 로드

In [ ]:
%pip install --quiet numpy pandas scipy scikit-learn xgboost lightgbm matplotlib seaborn

In [ ]:
from __future__ import annotations
import re, json, pickle, warnings, random
from pathlib import Path
from typing import Optional
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

NB1 = Path('outputs/nb1')
NB2 = Path('outputs/nb2')
OUT = Path('outputs/nb4'); OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
meta_full      = pd.read_csv(NB1 / 'meta_features_full.csv').set_index('dataset')
feature_groups = pd.read_csv(NB1 / 'feature_groups.csv')
brickts_wide   = pd.read_csv(NB2 / 'brickts_wide.csv')

print(f'meta_full {meta_full.shape} | brickts_wide {brickts_wide.shape}')
brickts_wide.head()

## Section 1 — USER CONFIG

`AGG_FUNC` 만 NB4 고유 설정. 나머지(split / model / HP / early stopping)는 **NB2 와 동일** 하게 두어야
"축 입력 제거" 외 변수가 통제되어 성능 하락을 공정하게 비교할 수 있다.

In [ ]:
# ===== USER CONFIG =====
# (A) 점수 집계 — 'mean'|'median' = 데이터셋당 1행, None = 집계 없이 combo 단위(개별 48 combo)
AGG_FUNC = None            # 'mean' | 'median' | None

# (B) 포함할 모델 축 one-hot. None(기본) = 축 전부 제외(메타피처만).
#     일부만 넣어 부분 ablation 가능: 예) ['Architecture'] → arch one-hot 만 추가.
#     축 one-hot 은 combo 단위가 있어야 의미 → AGG_FUNC=None 에서만 유효 (집계 모드면 무시).
INCLUDE_AXES = None        # None | ['Architecture','Scope','Level'] 중 일부

# (C) Feature subset (메타피처). None: 전체. 축은 (B) 로 별도 제어.
SELECTED_FEATURES: Optional[list] = None
SELECTED_GROUPS:   Optional[list] = None
SELECTED_PATTERN:  Optional[str]  = None
EXCLUDED_FEATURES: list           = []
EXCLUDED_PATTERN:  Optional[str]  = None

# (D) Targets / model / 학습 세팅 — NB2 와 동일 (모델 크기·학습 설정 비교 가능)
TARGET_COLS      = ['perf_score', 'time_score', 'mem_score']
PRIMARY_TARGET   = 'perf_score'
MODEL_NAME       = 'XGBoost'       # 'LightGBM' | 'XGBoost'
MODEL_CANDIDATES = [MODEL_NAME]
USE_GPU          = True
GPU_PARAMS       = {
    'LightGBM': {'device': 'gpu'},
    'XGBoost':  {'device': 'cuda', 'tree_method': 'hist'},
}
EARLY_STOPPING_ROUNDS = 50
DEFAULT_HP = {
    'LightGBM': {'n_estimators': 500, 'max_depth': 3, 'learning_rate': 0.05,
                 'num_leaves': 8, 'min_child_samples': 10},
    'XGBoost':  {'n_estimators': 500, 'max_depth': 3, 'learning_rate': 0.05,
                 'min_child_weight': 5, 'reg_alpha': 0.1, 'reg_lambda': 1.0},
}
# =======================
# 'None'/'none' 문자열로 적어도 실제 None 으로 정규화
if isinstance(AGG_FUNC, str) and AGG_FUNC.lower() == 'none':
    AGG_FUNC = None
if isinstance(INCLUDE_AXES, str) and INCLUDE_AXES.lower() == 'none':
    INCLUDE_AXES = None
print(f'AGG_FUNC={AGG_FUNC} | INCLUDE_AXES={INCLUDE_AXES} | MODEL={MODEL_NAME} | USE_GPU={USE_GPU}')

## Section 2 — NB2 split 로드 (동일 train/val/test)

NB4 가 split 을 재계산하지 않고, NB2 가 저장한 `best_model.pkl` 의 분할을 **그대로 로드**한다.
→ NB2 / NB3 와 **완전히 동일한 train/val/test** 보장 (§6 NB3 비교 ROC 가 같은 test 데이터에서 평가됨).

In [ ]:
with open(NB2 / 'best_model.pkl', 'rb') as f:
    _nb2_bundle = pickle.load(f)
TRAIN_DATASETS = list(_nb2_bundle['train_datasets'])
VAL_DATASETS   = list(_nb2_bundle['val_datasets'])
TEST_DATASETS  = list(_nb2_bundle['test_datasets'])
print(f'NB2 split 로드 — TRAIN {len(TRAIN_DATASETS)} | VAL {len(VAL_DATASETS)} | TEST {len(TEST_DATASETS)}')
print(f'TEST(inference holdout) : {TEST_DATASETS}')

## Section 3 — Feature subset + 학습 테이블

X = 메타피처 + (선택 시) `INCLUDE_AXES` 축 one-hot. `brickts_wide` 의 combo별 점수를 `AGG_FUNC` 에 따라 구성:
- `'mean'`/`'median'` → 데이터셋당 1행으로 집계 (축 컬럼 없어 `INCLUDE_AXES` 무시)
- `None` → 집계 없이 combo 단위 (한 데이터셋 = 48행) → `INCLUDE_AXES` 로 축 일부를 X 에 추가 가능

In [ ]:
def apply_feature_selection(meta_df, fg):
    feats = list(meta_df.columns)
    grp_map = dict(zip(fg['feature'], fg['group']))
    if SELECTED_FEATURES is not None:
        feats = [f for f in SELECTED_FEATURES if f in feats]
    elif SELECTED_GROUPS is not None:
        feats = [f for f in feats if grp_map.get(f) in SELECTED_GROUPS]
    elif SELECTED_PATTERN is not None:
        rx = re.compile(SELECTED_PATTERN)
        feats = [f for f in feats if rx.search(f)]
    feats = [f for f in feats if f not in set(EXCLUDED_FEATURES)]
    if EXCLUDED_PATTERN is not None:
        rx2 = re.compile(EXCLUDED_PATTERN)
        feats = [f for f in feats if not rx2.search(f)]
    return feats

selected_feats = apply_feature_selection(meta_full, feature_groups)
print(f'Selected meta features: {len(selected_feats)} / {meta_full.shape[1]}  (축 one-hot 미포함)')

In [ ]:
def build_table(wide, meta_df, feat_cols, agg_func):
    """agg_func='mean'|'median' → 데이터셋당 1행 집계. agg_func=None → combo 단위 그대로(집계 없음)."""
    no_agg = agg_func is None or (isinstance(agg_func, str) and agg_func.lower() == 'none')
    if no_agg:
        base = wide[['dataset', 'arch', 'scope', 'level'] + TARGET_COLS].copy()
    else:
        base = wide.groupby('dataset')[TARGET_COLS].agg(agg_func).reset_index()
    df = base.merge(meta_df[feat_cols].reset_index(), on='dataset', how='left')
    for c in feat_cols:
        df[c] = df[c].fillna(df[c].median())
    return df

DT_all = build_table(brickts_wide, meta_full, selected_feats, AGG_FUNC)
DT_all = DT_all.dropna(subset=[PRIMARY_TARGET]).reset_index(drop=True)
for t in [c for c in TARGET_COLS if c != PRIMARY_TARGET]:
    DT_all[t] = DT_all[t].fillna(0.0)

# 가능한 모든 축 one-hot 을 미리 생성 (combo 단위에서만 축 컬럼 존재). 조합 비교(§6)에서 재사용.
AXIS_MAP  = {'architecture': 'arch', 'arch': 'arch', 'scope': 'scope', 'level': 'level'}
ALL_AXES  = [a for a in ['arch', 'scope', 'level'] if a in DT_all.columns]
_oh_by_axis = {}
if ALL_AXES:
    oh = pd.get_dummies(DT_all[ALL_AXES].astype(str)).astype(int)
    DT_all = pd.concat([DT_all, oh], axis=1)
    _oh_by_axis = {a: [c for c in oh.columns if c.startswith(a + '_')] for a in ALL_AXES}


def axes_to_cols(axes):
    """INCLUDE_AXES(이름 리스트/None) → 해당 축 one-hot 컬럼들. 집계 모드면 빈 리스트."""
    cols = []
    for a in (axes or []):
        cols += _oh_by_axis.get(AXIS_MAP.get(str(a).lower(), ''), [])
    return cols


def axes_tag(axes):
    sel = [AXIS_MAP.get(str(a).lower()) for a in (axes or [])]
    sel = [a for a in sel if a in _oh_by_axis]
    return 'no-axis' if not sel else '+'.join(sel)


onehot_cols = axes_to_cols(INCLUDE_AXES)
X_COLS      = list(selected_feats) + onehot_cols
AXIS_TAG    = axes_tag(INCLUDE_AXES)
if INCLUDE_AXES and not onehot_cols:
    print(f'[!] INCLUDE_AXES={INCLUDE_AXES} 이지만 축 컬럼 없음(집계 모드?) → 무시')
DT_all.to_csv(OUT / 'dataset_level_table.csv', index=False)

# 학습은 train+val 만. test 의 실제값은 §5 추론에서 DT_all 에서 직접 조회.
_trainval = TRAIN_DATASETS + VAL_DATASETS
DT = DT_all[DT_all['dataset'].isin(_trainval)].reset_index(drop=True)
_unit = 'combo 단위(집계 없음)' if AGG_FUNC is None else f'데이터셋 단위({AGG_FUNC})'
print(f'AGG_FUNC={AGG_FUNC} → {_unit} | {AXIS_TAG} | DT_all {DT_all.shape} | DT(train+val) {DT.shape} | X_COLS {len(X_COLS)}')
DT[['dataset'] + TARGET_COLS].head()

## Section 4 — 학습 (train) + validation early stopping

NB2 와 동일한 `get_base_model` / `fit_with_early_stopping`. target 별 단일 모델을 val 로 early stopping 후
`MultiOutputRegressor` 로 묶는다.

In [ ]:
import xgboost as xgb
import lightgbm as lgb
from sklearn.multioutput import MultiOutputRegressor


def get_base_model(name, **params):
    if USE_GPU:
        params = {**GPU_PARAMS.get(name, {}), **params}
    if name == 'XGBoost':
        return xgb.XGBRegressor(random_state=42, n_jobs=-1, verbosity=0, **params)
    return lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1, **params)


def fit_with_early_stopping(name, params, X_tr, Y_tr, X_va, Y_va, target_cols, es_rounds):
    use_es = X_va is not None and len(X_va) > 0
    estimators, best_iters = [], {}
    for tgt in target_cols:
        if name == 'XGBoost':
            if use_es:
                est = get_base_model(name, early_stopping_rounds=es_rounds, **params)
                est.fit(X_tr, Y_tr[tgt], eval_set=[(X_va, Y_va[tgt])], verbose=False)
            else:
                est = get_base_model(name, **params); est.fit(X_tr, Y_tr[tgt])
            best_iters[tgt] = int(getattr(est, 'best_iteration', -1) or -1)
        else:
            est = get_base_model(name, **params)
            if use_es:
                est.fit(X_tr, Y_tr[tgt], eval_set=[(X_va, Y_va[tgt])],
                        callbacks=[lgb.early_stopping(es_rounds, verbose=False),
                                   lgb.log_evaluation(0)])
            else:
                est.fit(X_tr, Y_tr[tgt])
            best_iters[tgt] = int(getattr(est, 'best_iteration_', -1) or -1)
        estimators.append(est)
    mor = MultiOutputRegressor(get_base_model(name, **params))
    mor.estimators_ = estimators
    mor.n_features_in_ = X_tr.shape[1]
    return mor, best_iters

### 2-class 라벨 + target별 지표 함수 (NB2 와 동일)

`score < 0 → CD`, `score >= 0 → CI`. `mse/mae/spearman/sign_acc/accuracy/f1/auc` 를 한 번에 계산.

In [ ]:
from scipy.stats import spearmanr
from sklearn.metrics import (mean_absolute_error, f1_score, accuracy_score,
                             mean_squared_error, roc_auc_score)


def to_two_class(y):
    return np.where(y >= 0, 'CI', 'CD')


def per_target_metrics(y_true, y_pred):
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[m], y_pred[m]
    if len(y_true) < 2:
        nan = float('nan')
        return {'mse': nan, 'mae': nan, 'spearman': nan,
                'sign_acc': nan, 'accuracy': nan, 'f1': nan, 'auc': nan}
    sp, _ = spearmanr(y_true, y_pred)
    yt_bin = (y_true >= 0).astype(int)
    try:
        auc = float(roc_auc_score(yt_bin, y_pred)) if len(np.unique(yt_bin)) == 2 else float('nan')
    except ValueError:
        auc = float('nan')
    return {
        'mse':      float(mean_squared_error(y_true, y_pred)),
        'mae':      float(mean_absolute_error(y_true, y_pred)),
        'spearman': float(sp) if not np.isnan(sp) else 0.0,
        'sign_acc': float(np.mean(np.sign(y_true) == np.sign(y_pred))),
        'accuracy': float(accuracy_score(to_two_class(y_true), to_two_class(y_pred))),
        'f1':       float(f1_score(to_two_class(y_true), to_two_class(y_pred),
                                   average='macro', labels=['CD', 'CI'], zero_division=0)),
        'auc':      auc,
    }

In [ ]:
dt_tr = DT[DT['dataset'].isin(TRAIN_DATASETS)].reset_index(drop=True)
dt_va = DT[DT['dataset'].isin(VAL_DATASETS)].reset_index(drop=True)
X_tr, Y_tr = dt_tr[X_COLS].astype(float), dt_tr[TARGET_COLS]
X_va, Y_va = dt_va[X_COLS].astype(float), dt_va[TARGET_COLS]
print(f'train datasets {len(dt_tr)} | val datasets {len(dt_va)} (early stopping)')

MODEL_USED   = MODEL_CANDIDATES[0]
MODEL_PARAMS = DEFAULT_HP[MODEL_USED]
full_model, best_iters = fit_with_early_stopping(
    MODEL_USED, MODEL_PARAMS, X_tr, Y_tr, X_va, Y_va, TARGET_COLS, EARLY_STOPPING_ROUNDS)
print(f'Trained {MODEL_USED} (no-axis, GPU={USE_GPU}) | best_iter {best_iters}')

## Section 5 — TEST 데이터셋 추론

학습/검증과 겹치지 않는 **TEST 데이터셋**에 추론. no-axis 모델은 메타피처만 입력하므로 한 데이터셋의 모든 행에 **동일 예측**이 나온다.
`DT_all` 의 test 행에 그대로 예측을 붙여 비교 (집계 모드와 None 모드가 같은 코드 경로):
- `AGG_FUNC='mean'/'median'` → 데이터셋당 1행. 예측 1점 vs 집계 실제 1점.
- `AGG_FUNC=None` → 데이터셋당 48 combo 행. 동일 예측 vs 개별 combo 실제값 (집계로 뭉개지 않고 원본 분포와 직접 대조).

TEST 행 전체를 pooled 하여 target별 ROC / confusion / scatter 로 평가한다.

In [ ]:
# ===== USER CONFIG (Section 3) =====   ← NB3 §3 추론 설정과 동일 역할
# inference 대상 TEST 데이터셋 (train+val 과 절대 겹치지 않음). None = test 전부.
MILANO_INFER_DATASETS = None          # 예: ['Milano-6610', 'Milano-6373']
INCLUDE_STANDARD_IN_INFERENCE = False # NB3 pooled=test 와 동일하게 STANDARD 제외
STANDARD_DATASETS = ['ETTh1', 'ETTh2', 'ETTm1', 'ETTm2',
                     'Traffic', 'Electricity', 'Exchange', 'Weather', 'ILI']
# 시각화·지표에 쓸 TEST 데이터셋 부분집합. None = inference 전체.
VIZ_DATASETS = None                   # 예: ['Milano-6610']
# ====================================

_excluded = set(TRAIN_DATASETS) | set(VAL_DATASETS)   # 학습/검증 지역은 inference 금지
_bw = set(brickts_wide['dataset'].unique())
_avail = lambda d: d in _bw and d in meta_full.index and d not in _excluded

_holdout = [d for d in TEST_DATASETS if _avail(d)]
if MILANO_INFER_DATASETS is not None:
    _holdout = [d for d in _holdout if d in set(MILANO_INFER_DATASETS)]
_std = [d for d in STANDARD_DATASETS if _avail(d)] if INCLUDE_STANDARD_IN_INFERENCE else []
INFERENCE_DATASETS = _holdout + _std
print(f'Milano test (n={len(_holdout)}) : {_holdout}')
print(f'Standard    (n={len(_std)}) : {_std}')
print(f'INFERENCE_DATASETS (n={len(INFERENCE_DATASETS)}) : {INFERENCE_DATASETS}')

In [ ]:
# no-axis 모델은 메타피처만 입력 → 한 데이터셋의 모든 행에 동일 예측.
# DT_all 의 test 행에 그대로 예측을 붙임 (AGG_FUNC='mean'/'median'→1행, None→48 combo 행).
parts = []
for ds in INFERENCE_DATASETS:
    sub = DT_all[DT_all['dataset'] == ds]
    if sub.empty:
        continue
    yhat = np.asarray(full_model.predict(sub[X_COLS].astype(float)), dtype=float)
    out = sub[['dataset'] + TARGET_COLS].copy()
    for j, tgt in enumerate(TARGET_COLS):
        out[f'pred_{tgt}'] = yhat[:, j]
    parts.append(out)
inf_results = pd.concat(parts, ignore_index=True)
inf_results.to_csv(OUT / 'inference_test_results.csv', index=False)

# pooled per-target 지표 (inference 행 전체)
inf_metrics = pd.DataFrame([
    {'target': tgt, **per_target_metrics(inf_results[tgt].values, inf_results[f'pred_{tgt}'].values)}
    for tgt in TARGET_COLS
])
inf_metrics.to_csv(OUT / 'inference_test_metrics.csv', index=False)

_unit = 'combo' if AGG_FUNC is None else 'dataset'
print(f'inf_results {inf_results.shape} (1행 = 1 {_unit}; 예측 vs 실제)')
print('\nPooled per-target metrics (inference 전체):')
inf_metrics.set_index('target').round(3)

## Section 6 — 평가 시각화

① scatter(실제 vs 예측, main `INCLUDE_AXES`) · ② perf_score 2-class(CD/CI) confusion (VIZ_DS pooled).

③ **perf_score ROC — `INCLUDE_AXES` 조합별 비교**: `[['Architecture','Scope','Level'], ['Architecture'], ['Scope'], ['Level'], None]`
을 NB4 내부에서 각각 재학습해 **동일 TEST 데이터**로 비교한다.
(`['Architecture','Scope','Level']` = 전체 축 = NB2/NB3 with-axis 와 동등 → NB3 결과를 따로 불러올 필요 없음.)

In [ ]:
# 시각화 대상 데이터셋 (VIZ_DATASETS 부분집합) 선택
_all_ds = inf_results['dataset'].unique().tolist()
VIZ_DS = [d for d in (VIZ_DATASETS or _all_ds) if d in _all_ds]
_miss = [d for d in (VIZ_DATASETS or []) if d not in _all_ds]
if _miss:
    print('[!] inference 에 없는 데이터셋 무시:', _miss)
viz = inf_results[inf_results['dataset'].isin(VIZ_DS)]
_pt = 'combo' if AGG_FUNC is None else 'dataset'
print(f'VIZ_DS (n={len(VIZ_DS)}) : {VIZ_DS}  |  점 = 1 {_pt}, rows={len(viz)}')

# scatter — 실제 vs NB4 예측 (점 = dataset 또는 combo)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, tgt in zip(axes, TARGET_COLS):
    ax.scatter(viz[tgt], viz[f'pred_{tgt}'], alpha=0.6, s=45, color='steelblue')
    ax.plot([-1, 1], [-1, 1], 'k--', alpha=0.4)
    ax.axhline(0, color='gray', ls=':', alpha=0.5); ax.axvline(0, color='gray', ls=':', alpha=0.5)
    ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05)
    ax.set_xlabel(f'actual {tgt}'); ax.set_ylabel(f'pred {tgt}')
    ax.set_title(tgt)
plt.suptitle(f'TEST — 실제 vs NB4 ({AXIS_TAG}) 예측 (점 = 1 {_pt})', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve
LABELS = ['CD', 'CI']

# --- perf_score 2-class(CD/CI) confusion matrix (main INCLUDE_AXES, VIZ_DS pooled) ---
yt = to_two_class(viz[PRIMARY_TARGET].values)
yp = to_two_class(viz[f'pred_{PRIMARY_TARGET}'].values)
cm = confusion_matrix(yt, yp, labels=LABELS)
_f1 = f1_score(yt, yp, average='macro', labels=LABELS, zero_division=0)

fig, ax = plt.subplots(figsize=(5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=LABELS, yticklabels=LABELS, ax=ax)
ax.set_xlabel(f'Predicted (NB4 {AXIS_TAG})'); ax.set_ylabel('Actual (BrickTS)')
ax.set_title(f'{PRIMARY_TARGET} 2-class (CD/CI)  | VIZ_DS pooled\nacc={accuracy_score(yt, yp):.2f}  f1={_f1:.2f}')
plt.tight_layout(); plt.show()

# ===== perf_score ROC — INCLUDE_AXES 조합별 비교 (NB4 내부 재학습) =====
# ['Architecture','Scope','Level'] = 전체 축(= NB2/NB3 with-axis 와 동등). None = 축 전부 제외.
AXIS_COMPARE = [['Architecture', 'Scope', 'Level'], ['Architecture'], ['Scope'], ['Level'], None]


def _train_infer_perf(axes):
    """주어진 축 조합으로 재학습 후 TEST 추론 → perf_score (yt_bin, y_score)."""
    cols = list(selected_feats) + axes_to_cols(axes)
    dtr = DT_all[DT_all['dataset'].isin(TRAIN_DATASETS)]
    dva = DT_all[DT_all['dataset'].isin(VAL_DATASETS)]
    mdl, _ = fit_with_early_stopping(MODEL_USED, MODEL_PARAMS,
                                     dtr[cols].astype(float), dtr[TARGET_COLS],
                                     dva[cols].astype(float), dva[TARGET_COLS],
                                     TARGET_COLS, EARLY_STOPPING_ROUNDS)
    test = DT_all[DT_all['dataset'].isin(INFERENCE_DATASETS)]
    pidx = TARGET_COLS.index(PRIMARY_TARGET)
    ys = np.asarray(mdl.predict(test[cols].astype(float)), dtype=float)[:, pidx]
    yb = (test[PRIMARY_TARGET].values >= 0).astype(int)
    return yb, ys


if AGG_FUNC is not None:
    print('[!] AGG_FUNC 집계 모드 → 축 one-hot 불가, 모든 조합이 동일(no-axis). 조합 비교는 AGG_FUNC=None 권장.')

fig, ax = plt.subplots(figsize=(6.4, 6.0))
auc_rows = []
for axes in AXIS_COMPARE:
    yb, ys = _train_infer_perf(axes)
    if len(np.unique(yb)) < 2:
        continue
    fpr, tpr, _ = roc_curve(yb, ys)
    auc = roc_auc_score(yb, ys)
    ax.plot(fpr, tpr, lw=2.2, label=f'{axes_tag(axes)} (AUC={auc:.3f})')
    auc_rows.append({'axes': axes_tag(axes), 'auc': auc})
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate (CI=positive)')
ax.set_title(f'{PRIMARY_TARGET} ROC — INCLUDE_AXES 조합별 (동일 TEST, {len(INFERENCE_DATASETS)} ds)')
_handles, _labels = ax.get_legend_handles_labels()
plt.tight_layout(); plt.show()

# --- 범례만 별도 Figure ---
if _handles:
    figL = plt.figure(figsize=(5.6, 0.34 * len(_labels) + 0.6))
    figL.legend(_handles, _labels, loc='center', frameon=True, title=f'{PRIMARY_TARGET} axes (AUC)', fontsize=9)
    plt.axis('off'); plt.tight_layout(); plt.show()

# --- 조합별 AUC 막대 ---
auc_df = pd.DataFrame(auc_rows)
auc_df.to_csv(OUT / 'axis_ablation_auc.csv', index=False)
fig2, ax2 = plt.subplots(figsize=(7, 4.6))
sns.barplot(data=auc_df, x='axes', y='auc', hue='axes', legend=False, ax=ax2)
ax2.axhline(0.5, color='gray', ls='--', alpha=0.6)
ax2.set_ylim(0, 1.05); ax2.set_title(f'{PRIMARY_TARGET} AUC — INCLUDE_AXES 조합별 (동일 TEST)')
ax2.tick_params(axis='x', rotation=15)
plt.tight_layout(); plt.show()

## Section 7 — Target 별 feature importance (no-axis)

축 입력이 없을 때 어떤 메타피처가 CD/CI 우열을 가르는지. §4 의 `full_model` native gain importance.

In [ ]:
def native_importance(estimator, name, feats):
    if name == 'LightGBM':
        return np.asarray(estimator.booster_.feature_importance(importance_type='gain'), dtype=float)
    gains = estimator.get_booster().get_score(importance_type='gain')
    return np.asarray([gains.get(f, gains.get(f'f{i}', 0.0)) for i, f in enumerate(feats)], dtype=float)


imp_rows = []
for j, tgt in enumerate(TARGET_COLS):
    vals = native_importance(full_model.estimators_[j], MODEL_USED, X_COLS)
    rank = pd.Series(vals).rank(ascending=False, method='min').astype(int).values
    for f, v, r in zip(X_COLS, vals, rank):
        imp_rows.append({'model': MODEL_USED, 'target': tgt, 'feature': f,
                         'importance': float(v), 'rank': int(r)})
fi = pd.DataFrame(imp_rows)
fi.to_csv(OUT / 'feature_importance_noaxis.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, tgt in zip(axes, TARGET_COLS):
    sub = fi[fi['target'] == tgt].nlargest(15, 'importance')
    sns.barplot(data=sub, y='feature', x='importance', ax=ax, color='steelblue')
    ax.set_title(f'Top-15 — {tgt}')
plt.suptitle(f'{MODEL_USED} native gain importance (no-axis, 메타피처만)', y=1.01)
plt.tight_layout(); plt.show()

## Section 8 — 모델 bundle 저장

§4 에서 학습한 `full_model` 을 metadata(`axis_input=False`, `agg_func`, split)와 함께 `model_noaxis.pkl` 로 저장.

In [ ]:
bundle = {
    'model': full_model, 'model_name': MODEL_USED, 'targets': TARGET_COLS,
    'primary_target': PRIMARY_TARGET, 'hyperparams': MODEL_PARAMS,
    'feature_list': X_COLS, 'selected_meta_features': selected_feats,
    'agg_func': AGG_FUNC, 'include_axes': INCLUDE_AXES, 'onehot_cols': onehot_cols, 'axis_tag': AXIS_TAG,
    'train_datasets': TRAIN_DATASETS, 'val_datasets': VAL_DATASETS, 'test_datasets': TEST_DATASETS,
}
with open(OUT / 'model_noaxis.pkl', 'wb') as f:
    pickle.dump(bundle, f)
print(f'Saved model_noaxis.pkl ({MODEL_USED}, {AXIS_TAG}, agg={AGG_FUNC})')